# Lab 3 — Measuring RAG (EP11)

Since EP10 we've measured with **hit@3** alone: is a right page among the top three? — The book says to measure at **three levels** (§16.8)
1. **Retrieval:** did we retrieve the right chunks? — Recall@K · MRR · NDCG@K
2. **Generation:** is the answer backed by the documents, on topic, correct? — faithfulness · answer relevance · correctness
3. **End-to-end:** does the user get what they need? (needs people or real tasks — outside this lab)

**What you'll do**
1. Write Recall@K, MRR and NDCG@K ourselves, following Equations 16.22, 16.24 and 16.25
2. Measure **five systems** we've built since EP10 on the same 20 questions
3. Write faithfulness (Equation 16.26) and answer relevance (Equation 16.27) with an LLM judge
4. Measure three answers to the long question, and see what each metric sees — and what it doesn't

Every answer from the model comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §16.8 — CC BY-SA 4.0

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | the same key as Lab 1 (this lab costs about 7 US cents, mostly re-ranking) — how to set up the key, and why we chose the model, is in Lab 1 |
| Packages | `openai`, `python-dotenv`, `pypdf`, `numpy`, `tokenizers`, `requests` (installed in the next cell) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key in the code, and never share your `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv "pypdf>=6" numpy "tokenizers==0.23.3" requests

## 1. Everything we've built

The next three cells are code from earlier labs: three models · the 141-chunk library + BM25 · HyDE, RRF, re-ranking · the 20 test questions

In [ ]:
import os, re, math, json, urllib.request, requests
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from openai import OpenAI
from pypdf import PdfReader
from tokenizers import Tokenizer

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}
EMBED_MODEL = "qwen/qwen3-embedding-8b"
EMBED_EXTRA = {"provider": {"order": ["nebius"], "allow_fallbacks": False}}
RERANK_MODEL = "qwen/qwen3-reranker-8b"
RERANK_PROVIDER = {"order": ["fireworks"], "allow_fallbacks": False}
KEY = load_key()
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=KEY, max_retries=5)
COST = {"chat": 0.0, "embed": 0.0, "rerank": 0.0}

def ask(prompt):
    r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA,
                                       messages=[{"role": "user", "content": prompt}])
    COST["chat"] += r.usage.cost
    return r.choices[0].message.content

def embed(texts):
    vectors = []
    for i in range(0, len(texts), 64):
        r = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + 64],
                                     extra_body=EMBED_EXTRA)
        COST["embed"] += r.usage.cost
        vectors += [d.embedding for d in r.data]
    V = np.array(vectors)
    return V / np.linalg.norm(V, axis=1, keepdims=True)

def obj(**fields):
    return {"type": "object", "properties": fields,
            "required": list(fields), "additionalProperties": False}
TEXT = {"type": "string"}

def ask_json(prompt, schema):
    for attempt in range(2):                            # one retry if the JSON comes back cut off
        r = client.chat.completions.create(
            model=MODEL, temperature=0, extra_body=EXTRA, max_tokens=1500,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_schema",
                             "json_schema": {"name": "reply", "strict": True, "schema": schema}})
        COST["chat"] += r.usage.cost
        try:
            return json.loads(r.choices[0].message.content)
        except json.JSONDecodeError:
            pass
    raise ValueError("the model did not return valid JSON")

In [ ]:
urllib.request.urlretrieve("https://arxiv.org/pdf/2606.24937v1", "hitchhikers-guide-v1.pdf")
reader = PdfReader("hitchhikers-guide-v1.pdf")
HEADER = "H. Roitman — The Hitchhiker’s Guide to Agentic AI: From Foundations to Systems\n"
pages = [{"page": n, "text": reader.pages[n - 1].extract_text().replace(HEADER, "")}
         for n in range(292, 375)]                      # chapters 15–19
tok = Tokenizer.from_pretrained(MODEL)

def chunk_page(page, W=512, O=64):
    ids = tok.encode(page["text"], add_special_tokens=False).ids
    return [{"source": "hitchhikers-guide-v1.pdf", "page": page["page"],
             "text": tok.decode(ids[s:s + W])}
            for s in range(0, max(len(ids) - O, 1), W - O)]

chunks = [c for p in pages for c in chunk_page(p)]
E = embed([c["text"] for c in chunks])

def words(text):                                        # BM25 from EP10 Lab 2
    return re.findall(r"\w+", text.lower())
docs = [words(c["text"]) for c in chunks]
N, avgdl = len(docs), sum(len(d) for d in docs) / len(docs)
df = Counter(w for d in docs for w in set(d))
tfs = [Counter(d) for d in docs]
def idf(t):
    n = df.get(t, 0)
    return math.log((N - n + 0.5) / (n + 0.5) + 1)
def bm25_rank(query, k1=1.5, b=0.75):
    scores = [sum(idf(t) * tf[t] * (k1 + 1) / (tf[t] + k1 * (1 - b + b * len(d) / avgdl))
                  for t in words(query)) for d, tf in zip(docs, tfs)]
    return list(np.argsort(-np.array(scores)))
print(len(chunks), "chunks")

In [ ]:
def dense(query):                                       # EP10 Lab 1
    return list(np.argsort(-(E @ embed([query])[0])))

def rrf(lists, k=60):                                   # EP10 Lab 2
    score = {}
    for order in lists:
        for rank, i in enumerate(order, 1):
            score[i] = score.get(i, 0) + 1 / (k + rank)
    return sorted(score, key=score.get, reverse=True)

def hyde(query):                                        # Lab 1
    draft = ask("Write a short passage (about 100 words) from a textbook that answers this question. "
                "Write it as the textbook would, even if you are not sure.\n\nQuestion: " + query)
    return dense(draft)

def rerank(query, order, n=20):                         # Lab 1
    top = order[:n]
    r = requests.post("https://openrouter.ai/api/v1/rerank",
                      headers={"Authorization": f"Bearer {KEY}"},
                      json={"model": RERANK_MODEL, "provider": RERANK_PROVIDER, "query": query,
                            "documents": [chunks[i]["text"] for i in top]}, timeout=60)
    r.raise_for_status()
    data = r.json()
    COST["rerank"] += data["usage"]["cost"]
    return [top[x["index"]] for x in data["results"]]

TESTS = [  # (question, the pages that answer it) — from EP10 Lab 3
    ("What is PLAID?", {301}), ("What is HyDE?", {303}), ("What does FAISS stand for?", {298}),
    ("How can I search millions of vectors quickly?", {298}),
    ("Which method compares every query token with every document token?", {300}),
    ("How do I make keyword search understand synonyms?", {299}),
    ("When should I fine-tune a model instead of looking things up?", {296}),
    ("Which chunks should I give the model if small ones retrieve better but lack context?", {302, 303}),
    ("How do I combine keyword search and embedding search?", {298}),
    ("What analogy does the textbook use to explain RAG?", {295}),
    ("How does the textbook add randomness to retry waits?", {357}),
    ("What does FIFO mean for conversation history?", {345}),
    ("Where should the unchanging part of a prompt go so caching works?", {358}),
    ("When should a person approve an agent's action?", {355}),
    ("How does Graph RAG find groups of related entities?", {305}),
    ("Which chunking method splits where the topic changes?", {302}),
    ("Which frameworks are compared in Table 18.1?", {359}),
    ("What is the difference between episodic and semantic memory?", {321, 322}),
    ("What does an evaluator-optimizer workflow do?", {370, 371}),
    ("How does the harness notice an agent repeating the same tool call?", {357, 363}),
]

## 2. Retrieval metrics (§16.8.1)

$R_K$ = the chunks in the top K · $R^*$ = all the right chunks

$$\text{Recall@K}=\frac{|R_K\cap R^*|}{|R^*|}\;(16.22)\qquad \text{MRR}=\frac{1}{|Q|}\sum_i\frac{1}{\text{rank}_i}\;(16.24)\qquad \text{NDCG@K}=\frac{\text{DCG@K}}{\text{IDCG@K}},\;\text{DCG@K}=\sum_{i=1}^{K}\frac{rel_i}{\log_2(i+1)}\;(16.25)$$

Our **right chunks**: chunks from the pages that answer the question (page-level labels — one page can have two chunks) · `rel` is 0 or 1

In [ ]:
def relevant(gold):
    return {i for i, c in enumerate(chunks) if c["page"] in gold}

def recall_at_k(order, gold, k=5):
    R = relevant(gold)
    return len(set(order[:k]) & R) / len(R)

def reciprocal_rank(order, gold):
    for rank, i in enumerate(order, 1):
        if chunks[i]["page"] in gold:
            return 1 / rank
    return 0.0

def ndcg_at_k(order, gold, k=5):
    dcg = sum(1 / math.log2(rank + 1)
              for rank, i in enumerate(order[:k], 1) if chunks[i]["page"] in gold)
    ideal = sum(1 / math.log2(rank + 1) for rank in range(1, min(len(relevant(gold)), k) + 1))
    return dcg / ideal

In [ ]:
q, gold = TESTS[9]
order = dense(q)
print(q, gold, "→ pages", [chunks[i]["page"] for i in order[:5]])
print("Recall@5", recall_at_k(order, gold), "· RR", reciprocal_rank(order, gold),
      "· NDCG@5", round(ndcg_at_k(order, gold), 2))

In [ ]:
SYSTEMS = {
    "dense":           dense,
    "BM25":            bm25_rank,
    "hybrid (RRF)":    lambda q: rrf([dense(q), bm25_rank(q)]),
    "dense + re-rank": lambda q: rerank(q, dense(q)),
    "HyDE + re-rank":  lambda q: rerank(q, hyde(q)),
}

def measure(name, system):
    with ThreadPoolExecutor(5) as pool:
        orders = list(pool.map(system, [q for q, _ in TESTS]))
    golds = [g for _, g in TESTS]
    recall = np.mean([recall_at_k(o, g) for o, g in zip(orders, golds)])
    mrr = np.mean([reciprocal_rank(o, g) for o, g in zip(orders, golds)])
    ndcg = np.mean([ndcg_at_k(o, g) for o, g in zip(orders, golds)])
    print(f"{name:16} Recall@5 {recall:.2f}   MRR {mrr:.2f}   NDCG@5 {ndcg:.2f}")

for name, system in SYSTEMS.items():
    measure(name, system)

## 3. Generation metrics (§16.8.2)

$$\text{Faithfulness}=\frac{\#\,\text{claims supported by context}}{\#\,\text{total claims in answer}}\;(16.26)\qquad
\text{AnswerRelevance}=\frac{1}{N}\sum_{i=1}^{N}\cos\big(E(q),E(\hat q_i)\big)\;(16.27)$$

Both use an **LLM judge** — no right answer needed (reference-free), as in RAGAs (Listing 16.12).
We add **correctness**: compare with a reference answer (`ground_truth` in RAGAs)

In [ ]:
CLAIMS = obj(claims={"type": "array", "items": TEXT})
def claims_of(answer):
    return ask_json("Break this answer into short, standalone factual claims. If the answer only says that "
                    f"the context lacks the information, return an empty list.\n\nAnswer:\n{answer}", CLAIMS)["claims"]

SUPPORT = obj(reasoning=TEXT, supported={"type": "boolean"})
def supported(claim, docs):
    context = "\n\n".join(f"(page {d['page']}) {d['text']}" for d in docs)
    return ask_json(f"Context:\n{context}\n\nClaim: {claim}\n\nDoes the context support this claim?",
                    SUPPORT)["supported"]

def faithfulness(answer, docs):
    claims = claims_of(answer)
    if not claims:
        return None, []
    marks = [supported(c, docs) for c in claims]
    return sum(marks) / len(marks), list(zip(claims, marks))

Q3 = obj(q1=TEXT, q2=TEXT, q3=TEXT)
def answer_relevance(question, answer):
    qs = list(ask_json(f"Write three different questions that this answer would answer.\n\nAnswer:\n{answer}",
                       Q3).values())
    V = embed([question] + qs)
    return float(np.mean(V[1:] @ V[0]))

VERDICT = obj(reasoning=TEXT, correct={"type": "boolean"})
def correct(question, answer, reference):
    return ask_json(f"Question: {question}\nReference answer: {reference}\nAnswer: {answer}\n\n"
                    "Does the answer agree with the reference answer?", VERDICT)["correct"]

In [ ]:
question = ("In Roitman's book The Hitchhiker's Guide to Agentic AI, "
            "what analogy does the RAG chapter use to explain RAG?")
REFERENCE = "A library card: RAG gives a scholar who memorized a library years ago a card to look things up."
plain3 = [chunks[i] for i in dense(question)[:3]]
reranked3 = [chunks[i] for i in rerank(question, dense(question))[:3]]

SYSTEM_PROMPT = """You are a helpful assistant. Answer the question using ONLY
the provided context. If the context does not contain enough information,
say so explicitly. Cite your sources using [Doc N] notation."""
def rag_answer(query, docs):
    context = "\n\n".join(f"[Doc {i+1}] (Source: {c['source']}, Page: {c['page']})\n{c['text']}"
                          for i, c in enumerate(docs))
    return ask(f"{SYSTEM_PROMPT}\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:")

cases = {"no RAG":               (ask(question),                 reranked3),
         "RAG, plain top 3":     (rag_answer(question, plain3),    plain3),
         "RAG, re-ranked top 3": (rag_answer(question, reranked3), reranked3)}

In [ ]:
faith = {name: faithfulness(answer, docs) for name, (answer, docs) in cases.items()}
f, marks = faith["no RAG"]
for claim, ok in marks:
    print("✓" if ok else "✗", claim[:100])
print("faithfulness", round(f, 2))

In [ ]:
for name, (answer, docs) in cases.items():
    f, _ = faith[name]
    print(f"{name:21} faithfulness {'—' if f is None else f'{f:.2f}':>5}   "
          f"relevance {answer_relevance(question, answer):.2f}   "
          f"correct {correct(question, answer, REFERENCE)}")

In [ ]:
print({k: f"${v:.4f}" for k, v in COST.items()})

## Summary

| Metric | Answers the question | What we saw in the lab |
|---|---|---|
| Recall@5 | how many of the right chunks did we retrieve? | about 0.5 for almost every system — re-ranking doesn't find new chunks |
| MRR | how high is the first right chunk? | dense ≈ 0.5 → HyDE + re-rank ≈ 0.8 |
| NDCG@5 | are the right chunks near the top? | climbs like MRR |
| Faithfulness | is the answer backed by the documents? | no RAG ≈ 0.7–0.8 though its main claim is wrong · re-rank ≈ 1 · a refusal: unmeasurable or ≈ 0.7 · the judge can be wrong too |
| Answer relevance | is the answer on topic? | all three 0.7–0.9 — can't tell them apart |
| Correctness | is it right? (needs a reference answer) | only the re-ranked answer |

**No single metric is enough** — just as the book warns against optimizing one level

**Exercises**
1. Write **Precision@K** (Equation 16.23) and add it as a fourth column — which system scores highest?
2. Measure **Lab 2's agent** with faithfulness and correctness on 5 questions, against plain RAG — is it worth the extra model calls?
3. Try **RAGAs** (Listing 16.12) on the same three answers — are its numbers close to ours?

**Next — Lab 4:** build the same agent as Lab 2 with LangGraph